# Virginia Piedmont observation and identification timing

This notebook compares observation activity with overall identification activity in the Virginia Piedmont project.

Current operational definitions:
- `observed_count`: distinct observations grouped by `observed_on`.
- `identification_event_count`: all identification events grouped by identification timestamp.
- Delay: days from `observed_on` to each non-owner identification event (not only the first).

In [4]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import get_inat_session, get_observation_rows, summarize_time_series

PROJECT_ID = 'virginia-physiographic-regions-piedmont'
START = '2023-01-01'
END = None
PER_PAGE = 200
MAX_PAGES = 40

session = get_inat_session()

In [5]:
observations = get_observation_rows(
    project_id=PROJECT_ID,
    d1=START,
    d2=END,
    per_page=PER_PAGE,
    max_pages=MAX_PAGES,
    session=session,
)
observations['observed_on'] = pd.to_datetime(observations['observed_on'])
observations['all_identification_count'] = pd.to_numeric(observations['all_identification_count'], errors='coerce').fillna(0)
observations['non_owner_identification_count'] = pd.to_numeric(observations['non_owner_identification_count'], errors='coerce').fillna(0)

identification_events = observations[
    ['obs_id', 'iconic_taxon_name', 'observed_on', 'all_identification_timestamps', 'non_owner_identification_timestamps']
].copy()
identification_events = identification_events.explode('all_identification_timestamps')
identification_events = identification_events[identification_events['all_identification_timestamps'].notna()].copy()
identification_events['identification_at'] = pd.to_datetime(
    identification_events['all_identification_timestamps'],
    utc=True,
    errors='coerce',
)
identification_events = identification_events[identification_events['identification_at'].notna()].copy()
identification_events['identification_delay_days'] = (
    identification_events['identification_at'] - pd.to_datetime(identification_events['observed_on'], utc=True, errors='coerce')
).dt.total_seconds() / 86400.0

observations[['obs_id', 'observed_on', 'all_identification_count', 'iconic_taxon_name']].head()

TypeError: Session.request() got an unexpected keyword argument 'page'

In [ ]:
observed_ts = summarize_time_series(
    observations,
    date_col='observed_on',
    freq='M',
    count_name='observed_count',
    value_aggs={'total_identifications_on_observed_obs': ('all_identification_count', 'sum')},
)

identification_ts = summarize_time_series(
    identification_events,
    date_col='identification_at',
    freq='M',
    count_col=None,
    count_name='identification_event_count',
)

timeline = observed_ts.merge(identification_ts, on='period_start', how='outer').sort_values('period_start')
timeline[['observed_count', 'total_identifications_on_observed_obs', 'identification_event_count']] = timeline[
    ['observed_count', 'total_identifications_on_observed_obs', 'identification_event_count']
].fillna(0)
timeline['identification_events_per_observation'] = timeline['identification_event_count'] / timeline['observed_count'].replace(0, np.nan)
timeline['total_identifications_per_observation'] = timeline['total_identifications_on_observed_obs'] / timeline['observed_count'].replace(0, np.nan)
timeline.tail()

In [ ]:
counts_plot = timeline.melt(
    id_vars='period_start',
    value_vars=['observed_count', 'identification_event_count'],
    var_name='series',
    value_name='count',
)

fig = px.line(
    counts_plot,
    x='period_start',
    y='count',
    color='series',
    markers=True,
    title='Observation volume vs all identification-event volume',
)
fig.show()

ratio_plot = timeline.melt(
    id_vars='period_start',
    value_vars=['identification_events_per_observation', 'total_identifications_per_observation'],
    var_name='series',
    value_name='ratio',
)
ratio_fig = px.line(
    ratio_plot,
    x='period_start',
    y='ratio',
    color='series',
    markers=True,
    title='Identification rates per observation (all identifications)',
)
ratio_fig.show()

In [ ]:
delay_by_taxon = (
    identification_events.dropna(subset=['identification_delay_days'])
    .groupby('iconic_taxon_name', dropna=False)
    .agg(
        identification_event_count=('obs_id', 'size'),
        observations_represented=('obs_id', 'nunique'),
        mean_delay_days=('identification_delay_days', 'mean'),
        median_delay_days=('identification_delay_days', 'median'),
    )
    .reset_index()
    .sort_values(['identification_event_count', 'mean_delay_days'], ascending=[False, True])
)
delay_by_taxon

In [ ]:
delay_timeline = summarize_time_series(
    identification_events.dropna(subset=['identification_at', 'identification_delay_days']),
    date_col='identification_at',
    freq='M',
    group_cols=['iconic_taxon_name'],
    count_col=None,
    value_aggs={'mean_delay_days': ('identification_delay_days', 'mean')},
)
delay_timeline = delay_timeline[delay_timeline['count'] >= 25].copy()

delay_fig = px.line(
    delay_timeline,
    x='period_start',
    y='mean_delay_days',
    color='iconic_taxon_name',
    markers=True,
    title='Mean time to identification by iconic taxon (all identification events)',
)
delay_fig.show()